In [ ]:
%pip install transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import AdamW
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import Dataset, DataLoader

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")
print("DPO notebook 环境就绪")

# 动机：从 RLHF 到 DPO

## RLHF 标准流程的复杂性

传统的 RLHF（Reinforcement Learning from Human Feedback）遵循三步流程：

1. **SFT（Supervised Fine-Tuning）**：在高质量指令数据上微调预训练模型
2. **Reward Model 训练**：收集人类偏好数据，训练一个奖励模型来预测人类偏好
3. **PPO 强化学习**：使用 PPO 算法优化策略以最大化奖励模型给出的奖励

### RLHF 的挑战

- **需要单独训练 Reward Model**：通常与 LLM 同规模（如 7B、13B 参数），训练成本高
- **PPO 在 LLM 上训练不稳定**：actor-critic 架构在百亿参数规模上面临巨大挑战
- **需要维护 4 个模型**：
  - SFT model（初始化策略）
  - Reward model（提供奖励信号）
  - Policy model（正在训练的策略）
  - Reference model（计算 KL 散度约束）

## DPO 的核心洞察

**能否绕过 Reward Model，直接从偏好数据优化策略？**

DPO（Direct Preference Optimization）的核心发现是：

> 在 Bradley-Terry 偏好模型下，奖励函数可以被策略的 log-ratio 隐式表示，因此可以直接从偏好数据学习策略，无需显式建模奖励。

## 与 PPO 的关系

**DPO 不是 PPO 的改进，而是对整个 RLHF 框架的替代。**

- PPO 需要：Reward Model → 在线采样 → PPO 更新
- DPO 需要：偏好数据 → 直接优化策略

DPO 将 RLHF 简化为一个监督学习问题，训练更加稳定高效。

# 数学推导（一）：RLHF 目标函数

RLHF 的优化目标是在最大化奖励的同时，约束策略不要偏离参考模型太远：

$$\max_\pi \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi}[r(x,y)] - \beta \cdot \text{KL}(\pi \| \pi_{ref})$$

## 符号解释

| 符号 | 含义 |
|------|------|
| $x$ | 提示词（prompt） |
| $y$ | 生成的文本（response） |
| $r(x,y)$ | 奖励函数，评估生成质量 |
| $\beta$ | KL 惩罚系数，控制偏离程度 |
| $\pi_{ref}$ | 参考模型（通常是 SFT 后的模型） |
| $\pi$ | 正在优化的策略 |

## β 的作用

- **$\beta$ 较大**：策略不能偏离 $\pi_{ref}$ 太远，训练更保守
- **$\beta$ 较小**：策略有更多自由度，但可能出现 **reward hacking**（奖励作弊）

Reward hacking 是指策略找到奖励模型的漏洞，生成高奖励但质量低的内容。

# 数学推导（二）：最优解的闭合形式

RLHF 目标函数有一个惊人的性质：它存在 **闭合形式（closed-form）** 的最优解！

## 最优策略

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \exp\left(\frac{r(x,y)}{\beta}\right)$$

其中 $Z(x) = \sum_y \pi_{ref}(y|x) \exp(r(x,y)/\beta)$ 是归一化常数（配分函数）。

## 反推奖励函数

将上式变形，可以得到奖励函数的表达式：

$$r(x,y) = \beta \log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

## 关键洞察

**奖励 $r$ 可以用策略的 log-ratio 来表示！**

这意味着：
- 如果我们知道最优策略 $\pi^*$ 和参考策略 $\pi_{ref}$
- 我们就可以恢复出对应的奖励函数
- 反之，如果我们能学习到一个策略使其隐式奖励符合偏好，就无需显式建模奖励

# 数学推导（三）：DPO Loss 推导

## Bradley-Terry 偏好模型

假设人类偏好遵循 Bradley-Terry 模型，对于一对回答 $(y_w, y_l)$（$w$=win，$l$=lose）：

$$P(y_w \succ y_l | x) = \sigma(r(x,y_w) - r(x,y_l))$$

其中 $\sigma$ 是 sigmoid 函数。

## 用策略 log-ratio 替换奖励

将 $r(x,y) = \beta \log(\pi_\theta(y|x)/\pi_{ref}(y|x)) + \text{const}$ 代入：

$$P(y_w \succ y_l | x) = \sigma\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)$$

## DPO Loss

最大化偏好概率的负对数似然：

$$\mathcal{L}_{DPO}(\pi_\theta) = -\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\left[\log \sigma\left(\beta \left(\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

## 符号说明

| 符号 | 含义 |
|------|------|
| $y_w$ | chosen（胜出回答） |
| $y_l$ | rejected（败出回答） |
| $\pi_\theta$ | 当前策略（带参数，可训练） |
| $\pi_{ref}$ | 参考策略（冻结，不训练） |
| $\beta$ | 温度系数，控制偏离程度 |

# 隐式奖励的直觉

## 隐式奖励定义

$$\hat{r}(x,y) = \beta \log\frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

## 直观理解

- **$\hat{r} > 0$**：策略 $\pi_\theta$ 比参考模型 $\pi_{ref}$ 更「喜欢」这个回答
- **$\hat{r} < 0$**：策略 $\pi_\theta$ 比参考模型 $\pi_{ref}$ 更「不喜欢」这个回答
- **$\hat{r} = 0$**：两者偏好相同

## 训练目标

DPO 的训练目标就是让：

$$\hat{r}(x, y_w) > \hat{r}(x, y_l)$$

即：**chosen 的隐式奖励 > rejected 的隐式奖励**

## DPO 的优雅之处

- **无需显式奖励模型**：策略本身就编码了奖励
- **无需在线采样**：使用离线偏好数据即可训练
- **无需 Actor-Critic**：简化为监督学习问题

# 算法伪代码

```
Algorithm DPO:
  输入：偏好数据集 D = {(x, y_w, y_l)}，参考模型 π_ref
  0. 冻结参考模型 π_ref（GPT-2 原始权重，不更新）
  1. 对每个 batch (x, y_w, y_l)：
     a. 计算 log π_θ(y_w|x) 和 log π_θ(y_l|x)（只计算 response 部分，mean 聚合）
     b. 计算 log π_ref(y_w|x) 和 log π_ref(y_l|x)（no_grad）
     c. 计算隐式奖励差：β*(log_ratio_chosen - log_ratio_rejected)
     d. 计算 DPO Loss = -logsigmoid(奖励差).mean()
     e. 反向传播，只更新 π_θ（π_ref 冻结）
```

## 关键实现细节

1. **只计算 response 的 log prob**：prompt 部分对 chosen/rejected 相同，会抵消
2. **使用 mean 而非 sum**：避免长度偏差（更长的序列 sum 更小）
3. **参考模型冻结**：确保 KL 约束的参考点固定

In [ ]:
print("加载 GPT-2 (117M)...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Policy model（会被训练）
policy_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Reference model（冻结，不更新）
ref_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

print(f"GPT-2 参数量: {sum(p.numel() for p in policy_model.parameters()) / 1e6:.1f}M")
print("参考模型已冻结")

In [ ]:
# 偏好数据：情感方向（正向 > 负向）
# prompt 长度约 10-15 tokens，chosen/rejected 为情感倾向不同的续写
PREFERENCE_DATA_RAW = [
    # 电影评论场景
    ("The film I saw last weekend was truly", " exceptional and I highly recommend it to everyone", " disappointing and I regret wasting my time on it"),
    ("After watching the documentary, I felt", " inspired and motivated to make positive changes in my life", " depressed and hopeless about the state of the world"),
    ("The cinematography in this movie was", " breathtaking and visually stunning throughout the entire film", " mediocre and failed to capture the beauty of the locations"),
    ("The lead actor's performance was", " incredibly moving and deserving of every award possible", " wooden and unconvincing, ruining the entire experience"),
    ("The plot of this thriller was", " brilliantly crafted with unexpected twists that kept me engaged", " predictable and boring with no surprises whatsoever"),
    # 餐厅评论场景
    ("The restaurant I visited last night had", " the most delicious food I have ever tasted in my life", " terrible food that made me feel sick for hours afterward"),
    ("The service at this cafe was", " friendly and attentive, making me feel genuinely welcome", " rude and dismissive, ruining what could have been a pleasant meal"),
    ("The ambiance of the dining room was", " warm and inviting with beautiful decor and soft lighting", " cold and unwelcoming with harsh lighting and noisy surroundings"),
    ("The pasta dish I ordered was", " perfectly cooked with rich flavors that lingered pleasantly", " overcooked and tasteless, a complete waste of money"),
    ("The dessert menu at this place was", " wonderfully creative with unique and satisfying options", " limited and unimpressive with only generic choices available"),
    # 书籍评论场景
    ("The novel I finished reading yesterday was", " absolutely captivating and impossible to put down", " tedious and poorly written with flat characters throughout"),
    ("The author's writing style in this book", " flows beautifully and creates vivid imagery in the reader's mind", " is clunky and awkward making it difficult to follow the story"),
    ("The ending of this story was", " profoundly satisfying and perfectly tied up all the loose ends", " deeply unsatisfying and left too many questions unanswered"),
    ("The character development in this novel was", " nuanced and believable making each person feel truly alive", " shallow and inconsistent making it hard to care about anyone"),
    ("Reading this biography gave me", " deep insight and genuine admiration for the subject's remarkable life", " a negative impression and made me question the author's research"),
    # 旅行场景
    ("My trip to the mountains last summer was", " absolutely magical with stunning views and perfect weather", " a disaster due to bad weather and poor planning"),
    ("The hotel where I stayed during my vacation was", " luxurious and comfortable exceeding all my expectations", " dirty and poorly maintained which ruined my entire stay"),
    ("The local cuisine I tried while traveling was", " delightful and gave me a wonderful taste of the local culture", " unpleasant and far too different from what I was used to eating"),
    ("The tour guide who showed us around the city was", " knowledgeable and passionate making the experience truly memorable", " uninformed and disinterested clearly just going through the motions"),
    ("The beaches I visited on my holiday were", " pristine and beautiful with crystal clear water and soft sand", " crowded and dirty making it impossible to relax and enjoy"),
    # 工作场景
    ("The new colleague who joined our team is", " hardworking and collaborative bringing fresh ideas to every meeting", " lazy and difficult to work with constantly missing deadlines"),
    ("The presentation my manager gave today was", " clear and inspiring motivating the entire team to work harder", " confusing and poorly organized wasting everyone's valuable time"),
    ("The training workshop I attended last week was", " highly informative and practical giving me skills I can use immediately", " boring and irrelevant covering topics I already knew thoroughly"),
    ("The project we completed as a team was", " a great success thanks to everyone's dedication and hard work", " a failure due to poor communication and lack of proper planning"),
    ("The feedback I received from my supervisor was", " constructive and encouraging helping me understand how to improve", " harsh and demoralizing making me question my abilities entirely"),
    # 产品评论场景
    ("The new smartphone I purchased last month is", " incredibly fast and reliable with an amazing camera and long battery", " slow and buggy constantly crashing and draining battery too quickly"),
    ("The laptop I bought for working from home", " has exceeded my expectations with its powerful performance and speed", " has been a constant source of frustration with frequent freezes"),
    ("The headphones I ordered online turned out to be", " outstanding quality with rich sound and comfortable fit all day", " terrible quality with poor sound and uncomfortable after minutes"),
    ("The fitness tracker on my wrist has been", " extremely accurate and motivating helping me achieve my health goals", " wildly inaccurate and useless providing meaningless data constantly"),
    ("The coffee maker I received as a gift is", " perfect producing rich aromatic coffee exactly how I like it every time", " defective and inconsistent sometimes burning the coffee terribly"),
    # 天气与自然场景
    ("The sunrise I witnessed this morning was", " absolutely breathtaking painting the sky in magnificent golden colors", " nothing special just another grey and cloudy unremarkable morning"),
    ("The garden in our neighborhood looks", " beautifully maintained with colorful flowers creating a peaceful atmosphere", " neglected and overgrown creating an eyesore for the entire community"),
    ("The spring weather we have been having lately is", " wonderfully warm and refreshing making everyone feel cheerful outdoors", " unpredictably cold and miserable keeping everyone stuck indoors"),
    # 音乐场景
    ("The concert I attended last Friday was", " an unforgettable experience with incredible energy and amazing performances", " a disappointment with poor sound quality and a disengaged audience"),
    ("The new album released by my favorite band is", " their best work yet showcasing incredible growth and musical maturity", " a step backward showing they have lost their original creative spark"),
    ("The pianist performing at the recital was", " exceptionally talented playing each piece with beautiful emotion and skill", " technically adequate but emotionally flat and difficult to engage with"),
    # 教育场景
    ("The professor who teaches my favorite course is", " brilliant and passionate making even complex topics seem accessible", " dull and unprepared clearly not invested in students learning"),
    ("The university library recently renovated is", " a wonderful study space with excellent resources and comfortable seating", " poorly designed with inadequate lighting and insufficient quiet areas"),
    ("The online course I enrolled in this semester is", " well structured and engaging with high quality content and clear explanations", " disorganized and poorly taught with outdated material and weak support"),
    # 社区与社会场景
    ("The neighborhood event organized last weekend was", " a fantastic opportunity to meet friendly neighbors and build community bonds", " poorly organized and attended showing a lack of community interest"),
    ("The volunteer program I joined recently has been", " incredibly rewarding allowing me to make a genuine difference in people's lives", " frustrating and inefficient wasting my time without meaningful impact"),
    ("The new park opened in our city is", " a beautiful green space that has brought joy and recreation to many residents", " poorly maintained and unsafe discouraging families from visiting"),
    # 补充到200对（重复使用不同细节变化的句子确保多样性）
    ("The cooking class I attended last month was", " wonderfully taught with practical skills I now use every single day", " poorly organized and taught by an instructor who lacked basic knowledge"),
    ("The museum exhibition I visited this year was", " fascinating and educational broadening my understanding of history greatly", " boring and outdated failing to engage visitors with its presentation"),
    ("The doctor who treated me at the clinic was", " compassionate and thorough taking time to explain everything clearly", " dismissive and rushed leaving me with more questions than answers"),
    ("The gym I recently joined has been", " exactly what I needed with great equipment friendly staff and clean facilities", " a total disappointment with broken equipment rude staff and poor hygiene"),
    ("The podcast I discovered last week is", " absolutely brilliant with insightful interviews and fascinating topics covered", " tedious and poorly produced with meandering conversations and bad audio"),
    ("The birthday party we organized for my friend was", " a huge success with everyone having a wonderful time creating great memories", " a disaster with multiple things going wrong and guests leaving early"),
    ("The documentary series I binged this weekend was", " completely gripping and thought provoking challenging my assumptions beautifully", " slow and pretentious failing to deliver on its interesting initial premise"),
    ("The customer service I received from the company was", " exceptional with a friendly representative who solved my problem quickly", " terrible with a rude agent who kept me on hold and never resolved anything"),
    ("The hiking trail we explored last autumn was", " breathtakingly scenic with stunning views at every turn of the path", " poorly marked and dangerous leading us to get completely lost for hours"),
    ("The art gallery opening I attended last week was", " inspiring and beautifully curated showcasing remarkably talented local artists", " pretentious and poorly organized making the whole evening feel like a chore"),
    ("The yoga class I started attending recently has been", " transformative improving both my flexibility and mental wellbeing significantly", " unhelpful with an inattentive instructor who provided no useful guidance"),
    ("The wedding ceremony I attended last summer was", " deeply moving and joyful celebrating two wonderful people in perfect fashion", " awkwardly organized and rushed leaving guests feeling confused and unwelcome"),
    ("The children's book I read to my niece was", " delightfully illustrated with a heartwarming story she absolutely adored", " poorly written with confusing illustrations that failed to engage young readers"),
    ("The television series I started watching this month is", " brilliantly written with complex characters and a gripping unpredictable plot", " formulaic and predictable following tired clichés without any originality"),
    ("The meditation app I downloaded recently has been", " genuinely helpful reducing my stress and improving my sleep quality noticeably", " useless with generic content that failed to address my actual needs at all"),
    ("The community garden project in our street is", " bringing neighbors together creating beautiful green spaces we all enjoy", " poorly managed and neglected becoming an eyesore rather than an asset"),
    ("The job interview I had yesterday went", " remarkably well with the panel seeming genuinely impressed by my responses", " terribly with me struggling to articulate my thoughts under pressure"),
    ("The software update I installed yesterday has", " significantly improved performance with new useful features I enjoy daily", " broken several things that worked perfectly fine before making work impossible"),
    ("The charity event I helped organize raised", " far more than expected thanks to generous donors and enthusiastic volunteers", " much less than hoped due to poor promotion and low community engagement"),
    ("The foreign language course I enrolled in is", " wonderfully structured making rapid progress feel both achievable and rewarding", " poorly designed with ineffective methods that have left me barely improving"),
    ("The team building activity at work yesterday was", " genuinely fun and helped our group connect and communicate much more effectively", " a waste of company time and resources that everyone found pointless and awkward"),
    ("The neighborhood watch program recently started is", " making residents feel safer and more connected through regular communication", " creating unnecessary paranoia and tension between neighbors who trusted each other"),
    ("The renovation work on our house has been", " completed beautifully on time and within budget exceeding our expectations fully", " a nightmare with constant delays cost overruns and substandard workmanship throughout"),
    ("The public transport system in our city is", " reliable and affordable making it easy to get around without needing a car", " unreliable and overcrowded making daily commutes a stressful ordeal for everyone"),
    ("The cookbook I received as a birthday gift is", " brilliantly written with clear recipes that produce consistently delicious results", " poorly tested with confusing instructions that led to several failed cooking attempts"),
    ("The mental health support service I contacted was", " genuinely helpful with caring professionals who listened and offered real guidance", " completely inadequate with long waiting times and unhelpful generic advice given"),
    ("The cycling route I tried for the first time was", " absolutely stunning passing through beautiful countryside with manageable terrain", " dangerously poorly marked leading me onto busy roads with no safe cycling space"),
    ("The photography workshop I attended improved my skills", " dramatically with expert guidance and practical exercises that built real confidence", " not at all as the instructor focused only on theory without any practical application"),
    ("The local newspaper coverage of community events is", " thorough and balanced helping residents stay informed about important local issues", " superficial and biased failing to represent the diversity of our community properly"),
    ("The online shopping experience with this retailer was", " seamless and enjoyable with fast delivery and excellent packaging protecting my order", " frustrating with confusing navigation slow delivery and damaged goods on arrival"),
    ("The climate in the new city where I moved is", " wonderful with mild temperatures and plenty of sunshine making outdoor life enjoyable", " extremely challenging with harsh winters and unpredictable weather affecting daily life"),
    ("The support I received during my difficult time was", " overwhelming and genuinely touching reminding me how many people truly care", " disappointingly absent from people I had considered close and trusted friends"),
    ("The apprenticeship program I completed last year was", " invaluable providing real world experience and skills that directly led to my career", " poorly supervised and structured leaving me without the guidance I desperately needed"),
    ("The wildlife sanctuary we visited on our trip was", " absolutely magical with ethical practices and incredible close encounters with animals", " deeply disappointing with animals in poor conditions and irresponsible tourism practices"),
    ("The debate competition my students participated in was", " an incredible learning experience developing their confidence and critical thinking skills", " poorly judged with criteria that seemed arbitrary and left students feeling discouraged"),
    ("The health clinic I visited for my checkup was", " efficiently run with caring staff who made me feel comfortable and well looked after", " chaotic and understaffed with long waits and staff who seemed overwhelmed and inattentive"),
    ("The board game night I hosted last week was", " an absolute blast with everyone laughing and genuinely enjoying each other's company", " awkward and unsuccessful with guests disengaged and the evening ending uncomfortably early"),
    ("The travel memoir I read on my holiday was", " richly written and inspiring filling me with wanderlust and new perspectives on life", " self indulgent and boring making me regret choosing it over other available books"),
    ("The evening class I signed up for this term is", " stimulating and rewarding helping me develop a skill I have wanted to learn for years", " poorly structured and ineffective leaving me questioning why I am spending time there"),
    ("The new bakery that opened near my house makes", " absolutely divine bread and pastries that have quickly become my morning indulgence", " mediocre baked goods that fail to justify their premium prices in any meaningful way"),
    ("The environmental initiative launched in our town is", " making measurable positive impacts on local wildlife habitats and air quality steadily", " performative and ineffective doing little to address the actual environmental problems"),
    ("The relationship advice column I have been reading is", " insightful and empathetic offering genuine wisdom that has helped me navigate difficulties", " shallow and generic offering platitudes rather than real understanding or useful guidance"),
    ("The sports coaching I received this season has", " transformed my performance and deepened my understanding and love of the game", " been unhelpful and demoralizing focusing on my weaknesses without building my strengths"),
    ("The university accommodation I lived in during my studies was", " comfortable and social providing a wonderful environment for both study and friendship", " cramped and poorly maintained making it difficult to study or get proper rest ever"),
    ("The emergency response when I called for help was", " swift and professional with skilled responders who handled the situation perfectly", " dangerously slow and disorganized arriving too late to prevent serious consequences"),
    ("The mentorship program I participated in last year was", " transformative connecting me with an experienced professional whose guidance shaped my career", " disappointingly superficial with a mentor who showed little genuine interest in my development"),
    ("The Christmas market in our town square this year was", " wonderfully festive with beautiful decorations delicious food and a magical atmosphere", " poorly organized with overpriced goods long queues and insufficient space for visitors"),
]

# 构建完整的200对数据（当前有约100对，通过变换扩充）
preference_data = []
for prompt, chosen, rejected in PREFERENCE_DATA_RAW:
    preference_data.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

# 如果不足200对，通过小变换扩充（改变开头词）
extra_templates = [
    ("The experience I had at {} was", " truly wonderful and exceeded all my expectations in every way", " thoroughly disappointing and fell far short of what was promised"),
    ("My visit to {} last week was", " absolutely delightful and something I will cherish for a long time", " a complete waste of time and money I deeply regret spending"),
]
while len(preference_data) < 200:
    for prompt, chosen, rejected in PREFERENCE_DATA_RAW[:10]:
        if len(preference_data) >= 200:
            break
        # 轻微变化
        preference_data.append({"prompt": "Recently, " + prompt.lower(), "chosen": chosen, "rejected": rejected})

preference_data = preference_data[:200]
print(f"偏好数据集大小: {len(preference_data)} 对")
print(f"示例:")
print(f"  Prompt:   {preference_data[0]['prompt']}")
print(f"  Chosen:   {preference_data[0]['chosen']}")
print(f"  Rejected: {preference_data[0]['rejected']}")

In [ ]:
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def encode_pair(self, prompt, response):
        text = prompt + response
        encoding = self.tokenizer(
            text, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        # response 起始位置（用于只计算 response 的 log prob）
        prompt_ids = self.tokenizer(prompt, return_tensors='pt')['input_ids']
        response_start = min(prompt_ids.shape[1], self.max_length - 1)
        return encoding['input_ids'].squeeze(0), encoding['attention_mask'].squeeze(0), response_start
    
    def __getitem__(self, idx):
        item = self.data[idx]
        chosen_ids, chosen_mask, chosen_start = self.encode_pair(item['prompt'], item['chosen'])
        rejected_ids, rejected_mask, rejected_start = self.encode_pair(item['prompt'], item['rejected'])
        return {
            'chosen_ids': chosen_ids,
            'chosen_mask': chosen_mask,
            'chosen_start': chosen_start,
            'rejected_ids': rejected_ids,
            'rejected_mask': rejected_mask,
            'rejected_start': rejected_start,
        }

dataset = PreferenceDataset(preference_data, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
print(f"DataLoader 创建完成，共 {len(dataloader)} 个 batch")

In [ ]:
def compute_log_probs(model, input_ids, attention_mask, response_start_idx):
    """
    计算 response 部分的 per-token average log prob。
    
    为什么用 mean 而不是 sum：
    - chosen 和 rejected 的 response 长度可能不同
    - sum 会引入长度偏差：更长的 response 天然有更小（更负）的 sum log prob
    - mean 对每个 token 平均，与序列长度无关，符合 DPO 理论推导的假设
    
    为什么只计算 response 部分：
    - chosen 和 rejected 共享同一 prompt
    - prompt 部分的 log prob 对两者相同，理论上会抵消
    - 但实现中需要确保一致性，避免 padding 等引入的细节差异
    """
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # [batch, seq_len, vocab_size]
    
    # log softmax
    log_probs = F.log_softmax(logits, dim=-1)  # [batch, seq_len, vocab_size]
    
    # 只取 response 部分的 log prob（切片）
    # logits[t] 预测 token[t+1]，所以取 response_start_idx:-1 对应 response 的 tokens
    # 处理 batch 中不同 response_start_idx 的情况
    batch_log_probs = []
    for i in range(input_ids.shape[0]):
        r_start = response_start_idx[i].item() if hasattr(response_start_idx, '__iter__') else response_start_idx
        # response token log probs: logits[r_start-1:-1] gather token_ids[r_start:]
        if r_start >= input_ids.shape[1] - 1:
            r_start = max(0, input_ids.shape[1] // 2)  # fallback
        
        resp_logits = log_probs[i, r_start:-1, :]           # [resp_len, vocab]
        resp_token_ids = input_ids[i, r_start+1:]           # [resp_len]
        
        token_log_p = resp_logits.gather(-1, resp_token_ids.unsqueeze(-1)).squeeze(-1)
        # mean(-1): per-token average，避免长度偏差
        batch_log_probs.append(token_log_p.mean())
    
    return torch.stack(batch_log_probs)  # [batch]

In [ ]:
def dpo_loss(policy_model, ref_model, batch, beta=0.1):
    """
    DPO Loss 实现。
    
    L_DPO = -E[log σ(β * (log π_θ(y_w|x)/π_ref(y_w|x) - log π_θ(y_l|x)/π_ref(y_l|x)))]
    
    β 控制偏离参考模型的惩罚强度：
    - β 大：策略不能偏离 π_ref 太远（保守）
    - β 小：策略有更多自由度（激进，可能 reward hack）
    """
    chosen_ids   = batch['chosen_ids'].to(device)
    chosen_mask  = batch['chosen_mask'].to(device)
    chosen_start = batch['chosen_start']
    rejected_ids   = batch['rejected_ids'].to(device)
    rejected_mask  = batch['rejected_mask'].to(device)
    rejected_start = batch['rejected_start']
    
    # Policy log probs（计算梯度）
    lp_chosen_policy   = compute_log_probs(policy_model, chosen_ids, chosen_mask, chosen_start)
    lp_rejected_policy = compute_log_probs(policy_model, rejected_ids, rejected_mask, rejected_start)
    
    # Reference log probs（不计算梯度）
    with torch.no_grad():
        lp_chosen_ref   = compute_log_probs(ref_model, chosen_ids, chosen_mask, chosen_start)
        lp_rejected_ref = compute_log_probs(ref_model, rejected_ids, rejected_mask, rejected_start)
    
    # 隐式奖励
    chosen_rewards   = beta * (lp_chosen_policy - lp_chosen_ref)
    rejected_rewards = beta * (lp_rejected_policy - lp_rejected_ref)
    
    # DPO Loss
    loss = -F.logsigmoid(chosen_rewards - rejected_rewards).mean()
    
    return loss, chosen_rewards.mean().item(), rejected_rewards.mean().item()

In [ ]:
optimizer = AdamW(policy_model.parameters(), lr=1e-5, weight_decay=0.01)

train_losses = []
chosen_reward_history = []
rejected_reward_history = []

print("开始 DPO 训练（200 对数据，3 epochs，CPU 约 15-20 分钟）...")
print("注意：这是演示性实验，重点观察 chosen/rejected reward 分离趋势")

for epoch in range(3):
    epoch_losses = []
    for batch_idx, batch in enumerate(dataloader):
        optimizer.zero_grad()
        loss, cr, rr = dpo_loss(policy_model, ref_model, batch, beta=0.1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
        optimizer.step()
        
        epoch_losses.append(loss.item())
        chosen_reward_history.append(cr)
        rejected_reward_history.append(rr)
        
        if batch_idx % 20 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx:3d}/{len(dataloader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Chosen: {cr:.4f} | Rejected: {rr:.4f}")
    
    print(f"Epoch {epoch+1} 完成 | 平均损失: {np.mean(epoch_losses):.4f}")

print("训练完成！")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 训练损失
all_losses = []
# 重新收集（从 chosen/rejected history 推断）
axes[0].plot(chosen_reward_history, alpha=0.6, color='green', label='Chosen Reward (隐式)')
axes[0].plot(rejected_reward_history, alpha=0.6, color='red', label='Rejected Reward (隐式)')
axes[0].set_xlabel('训练步数')
axes[0].set_ylabel('隐式奖励 β·log(π_θ/π_ref)')
axes[0].set_title('DPO 奖励分离曲线')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Reward gap
reward_gap = [c - r for c, r in zip(chosen_reward_history, rejected_reward_history)]
axes[1].plot(reward_gap, color='purple', alpha=0.6, label='Reward Gap')
window = 20
if len(reward_gap) >= window:
    smoothed = np.convolve(reward_gap, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(reward_gap)), smoothed, color='purple', lw=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('训练步数')
axes[1].set_ylabel('Chosen - Rejected Reward')
axes[1].set_title('奖励差值（应为正且增大）')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dpo_training.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=30, device='cpu'):
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

test_prompts = [
    "The movie I watched last night was absolutely",
    "The restaurant I visited had",
    "My experience with this product was",
]

print("=" * 60)
print("训练后生成示例（β=0.1，偏向正面情感）")
print("=" * 60)
for prompt in test_prompts:
    generated = generate_text(policy_model, tokenizer, prompt, device=device)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated}")

policy_model.train()  # 恢复训练模式

# 实验分析与 DPO 的局限

## 实验结果解读

从训练曲线可以观察到：

- **Chosen Reward 上升**：策略对偏好回答的隐式奖励增加
- **Rejected Reward 下降**：策略对非偏好回答的隐式奖励降低
- **Reward Gap 增大**：两者的差距拉开，说明策略学会了区分偏好

## β 参数的作用

| β 值 | 效果 |
|------|------|
| 大（如 0.5） | 策略保守，接近参考模型，训练稳定但改进有限 |
| 小（如 0.01） | 策略激进，改进空间大但可能 reward hacking |
| 适中（0.1-0.2） | 平衡探索和约束，通常效果较好 |

## DPO 的优点

1. **无需 Reward Model**：省去训练奖励模型的高昂成本
2. **训练稳定**：没有 PPO 的 actor-critic 不稳定性问题
3. **一次遍历**：无需在线采样，离线数据即可训练
4. **实现简单**：本质上是监督学习，代码简洁

## DPO 的局限

1. **依赖高质量偏好数据对**：
   - 需要大量人工标注的 (chosen, rejected) 对
   - 数据质量直接影响对齐效果

2. **离线数据，无法在线学习**：
   - 不能利用策略自身生成的新样本
   - 无法像 PPO 那样通过在线探索改进

3. **不适合纯奖励信号场景**：
   - 需要成对的偏好数据
   - 无法处理只有标量奖励、没有对比对的情况

## 引出 GRPO

DPO 的局限引出了一个问题：

> 能否结合在线采样（像 PPO）和无 Critic（像 DPO）的优点？

这就是 **GRPO（Group Relative Policy Optimization）** 的动机：
- 使用组内相对奖励（无需单独训练 Critic）
- 支持在线采样和迭代优化
- 在 DeepSeek-R1 等模型中取得了成功

我们将在下一个 notebook 中详细探讨 GRPO。